In [1]:
import os
import numpy as np
import torch
import argparse
from minimodel import data
from minimodel import model_builder
from minimodel import model_trainer
from minimodel import metrics


In [2]:
# setup
device = torch.device('cuda')
mouse_id = 0
data_path = '../data'
weight_path = './checkpoints_16-320'
results_path = './results_16-320_test'
os.makedirs(weight_path, exist_ok=True)
np.random.seed(1)

# load images
img = data.load_images(data_path, mouse_id, file=data.img_file_name[mouse_id])

# load neurons
fname = '%s_nat60k_%s.npz'%(data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
spks, istim_train, istim_test, xpos, ypos, spks_rep_all = data.load_neurons(file_path = os.path.join(data_path, fname), mouse_id = mouse_id)
n_stim, n_neurons = spks.shape
print("spks_rep_all: ", spks_rep_all.shape)
print("spks: ", spks.shape)
# split train and validation set
itrain, ival = data.split_train_val(istim_train, train_frac=0.9)

# normalize data
spks, spks_rep_all = data.normalize_spks(spks, spks_rep_all, itrain)


ineur = np.arange(0, n_neurons) #np.arange(0, n_neurons, 5)
spks_train = torch.from_numpy(spks[itrain][:,ineur]).to(device)
spks_val = torch.from_numpy(spks[ival][:,ineur]).to(device)

print('spks_train: ', spks_train.shape, spks_train.min(), spks_train.max())
print('spks_val: ', spks_val.shape, spks_val.min(), spks_val.max())

img_train = torch.from_numpy(img[istim_train][itrain]).to(device).unsqueeze(1) # change :130 to 25:100 
img_val = torch.from_numpy(img[istim_train][ival]).to(device).unsqueeze(1)
img_test = torch.from_numpy(img[istim_test]).to(device).unsqueeze(1)

print('img_train: ', img_train.shape, img_train.min(), img_train.max())
print('img_val: ', img_val.shape, img_val.min(), img_val.max())
print('img_test: ', img_test.shape, img_test.min(), img_test.max())

input_Ly, input_Lx = img_train.shape[-2:]


seed = 1


raw image shape:  (68000, 66, 264)
cropped image shape:  (68000, 66, 130)
img:  (68000, 66, 130) -2.062947 2.088608 float32

loading activities from ../data/L1_A5_nat60k_2023_02_27.npz
spks_rep_all:  (500,)
spks:  (27533, 6636)

splitting training and validation set...
itrain:  (24779,)
ival:  (2754,)

normalizing neural data...
finished
spks_train:  torch.Size([24779, 6636]) tensor(0., device='cuda:0', dtype=torch.float64) tensor(58.7231, device='cuda:0', dtype=torch.float64)
spks_val:  torch.Size([2754, 6636]) tensor(0., device='cuda:0', dtype=torch.float64) tensor(49.2309, device='cuda:0', dtype=torch.float64)
img_train:  torch.Size([24779, 1, 66, 130]) tensor(-2.0629, device='cuda:0') tensor(2.0886, device='cuda:0')
img_val:  torch.Size([2754, 1, 66, 130]) tensor(-2.0629, device='cuda:0') tensor(2.0886, device='cuda:0')
img_test:  torch.Size([500, 1, 66, 130]) tensor(-2.0629, device='cuda:0') tensor(2.0886, device='cuda:0')


In [8]:
print(type(spks_rep_all[0]))
print(spks_rep_all[0].shape)

<class 'numpy.ndarray'>
(11, 6636)


In [3]:
# Building Model

nlayers = 2
nconv1 = 16
nconv2 = 320
model, in_channels = model_builder.build_model(NN=len(ineur), n_layers=nlayers, n_conv=nconv1, n_conv_mid=nconv2)
model_name = model_builder.create_model_name(data.mouse_names[mouse_id], data.exp_date[mouse_id], n_layers=nlayers, in_channels=in_channels)

model_path = os.path.join(weight_path, model_name)
print('model path: ', model_path)
model = model.to(device)

# Training the model
print(device)
if not os.path.exists(model_path):
    best_state_dict = model_trainer.train(model, spks_train, spks_val, img_train, img_val, device=device)
    torch.save(best_state_dict, model_path)
    print('saved model', model_path)
model.load_state_dict(torch.load(model_path))
print('loaded model', model_path)


core shape:  torch.Size([1, 320, 33, 65])
input shape of readout:  (320, 33, 65)
model name:  l1a5_022723_2layer_16_320_clamp_norm_depthsep_pool.pt
model path:  ./checkpoints_16-320/l1a5_022723_2layer_16_320_clamp_norm_depthsep_pool.pt
cuda
loaded model ./checkpoints_16-320/l1a5_022723_2layer_16_320_clamp_norm_depthsep_pool.pt


/tmp/ipykernel_2091283/2670656202.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


In [ ]:
print("spks_rep_all: ", spks_rep_all.shape)
print("spks_rep_all example: ", spks_rep_all[0].shape)
# test model
test_pred = model_trainer.test_epoch(model, img_test)
print('test_pred: ', test_pred.shape, test_pred.min(), test_pred.max())


test_fev, test_feve = metrics.feve(spks_rep_all, test_pred)
print('FEVE (test, all): ', np.mean(test_feve))

threshold = 0.15
print(f'filtering neurons with FEV > {threshold}')
valid_idxes = np.where(test_fev > threshold)[0]
print(f'valid neurons: {len(valid_idxes)} / {len(test_fev)}')
print(f'FEVE (test, FEV>0.15): {np.mean(test_feve[test_fev > threshold])}')

print( "---------")
print("spks_rep_all: ", spks_rep_all.shape)


spks_rep_all:  (500,)
test_pred:  (500, 6636) 0.0016735196 8.521689
FEVE (test, all):  0.59348226
filtering neurons with FEV > 0.15
valid neurons: 4242 / 6636
FEVE (test, FEV>0.15): 0.6821607947349548
---------
spks_rep_all:  (500,)


In [5]:
"""
# ---- Saving performance scores ----
file_name = "results_" + str(mouse_id)
results_file_path = os.path.join(results_path, file_name)

print(f"Results saved at: {results_file_path}")
np.savez(results_file_path, FEV_scores=test_fev, FEVE_scores=test_feve, neurons_index=ineur)
"""

'\n# ---- Saving performance scores ----\nfile_name = "results_" + str(mouse_id)\nresults_file_path = os.path.join(results_path, file_name)\n\nprint(f"Results saved at: {results_file_path}")\nnp.savez(results_file_path, FEV_scores=test_fev, FEVE_scores=test_feve, neurons_index=ineur)\n'